<a href="https://colab.research.google.com/github/IsaganiJulian/Guardian-Recruit-Fraud-Detection/blob/feature%2Foutlier-modeling-week4-kusuma/03_outlier_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Outlier Detection for Job Fraud (Stream B)

## Objective
The goal of this notebook is to detect fraudulent job postings using anomaly detection techniques such as Isolation Forest and Local Outlier Factor (LOF).

## 1. Load Data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [34]:
os.listdir('/content/drive/MyDrive/DTSC 5082- Group 12 - Guardian Recruit')

['data',
 'models',
 'notebooks',
 'src',
 'Project Road  Map and Planning',
 'Reports and Papers']

In [4]:
os.listdir('/content/drive/MyDrive/DTSC 5082- Group 12 - Guardian Recruit/data')

['raw', 'preprocessed', 'external']

In [5]:
os.listdir('/content/drive/MyDrive/DTSC 5082- Group 12 - Guardian Recruit/data/preprocessed')

['val.csv', 'train.csv', 'test.csv', 'train_clean_v1.csv']

In [6]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/DTSC 5082- Group 12 - Guardian Recruit/data/preprocessed/train_clean_v1.csv')
df.head()

,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,in_balanced_dataset,country,desc_len
0,Inside Sales Consultant B2B software company,"US, GA, Atlanta",Sales,35000-150000,<p>Katapult Group - We leverage technology and...,<p>Katapult Group is a global business buildin...,<p>Although this product converts extremly hig...,<p><b>Compensation:</b> 3000-5000$ per month p...,0,1,1,Full-time,Entry level,Certification,Internet,Sales,0,0,US,1907
1,Cad Designer,"US, OH, Cleveland",NaN,NaN,<p>We Provide Full Time Permanent Positions fo...,<p><b><i>(We have more than 1500 Job openings ...,Unspecified,Unspecified,0,0,0,Full-time,NaN,NaN,NaN,NaN,0,0,US,2505
2,Micro-grid Systems Engineer,"US, CA, Santa Monica",tech,NaN,<p>hello world</p>\r\n<p>talents23_ drives the...,<p>We have extensive experience in battery sto...,<ul>\r\n<li>Experience with utility interactiv...,"<p>Want to be part of a fast growing, high ene...",0,1,1,Full-time,NaN,NaN,NaN,NaN,0,0,US,699
3,Software Engineer,"US, NY, Brooklyn",Engineering,NaN,<p>Maker’s Row is an online marketplace that c...,<p>Our Engineering team values software qualit...,<ul>\r\n<li>3+ years of full-stack development...,<ul>\r\n<li>Healthcare</li>\r\n</ul><ul>\r\n<l...,0,1,1,Full-time,Associate,Bachelor's Degree,Computer Software,Engineering,0,0,US,1162
4,Messenger Courier - Part Time,"US, DC, Washington",NaN,NaN,"<p>Novitex Enterprise Solutions, formerly Pitn...",<p>The Messenger Courier will be based in our ...,<p><b>Minimum Requirements:</b></p>\r\n<ul>\r\...,Unspecified,0,1,0,Part-time,Entry level,High School or equivalent,Government Administration,Customer Service,0,0,US,1489


In [7]:
df.columns

Index(['title', 'location', 'department', 'salary_range', 'company_profile',
       'description', 'requirements', 'benefits', 'telecommuting',
       'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent', 'in_balanced_dataset', 'country', 'desc_len'],
      dtype='object')

## 2. Feature Engineering

We convert raw data into numerical features suitable for machine learning models.

In [8]:
# Handle missing values

df['employment_type'] = df['employment_type'].fillna('Unknown')
df['required_education'] = df['required_education'].fillna('Unknown')

In [9]:
df.columns

Index(['title', 'location', 'department', 'salary_range', 'company_profile',
       'description', 'requirements', 'benefits', 'telecommuting',
       'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent', 'in_balanced_dataset', 'country', 'desc_len'],
      dtype='object')

In [10]:
# Encode categorical variables

df['employment_type'] = df['employment_type'].astype('category').cat.codes
df['required_education'] = df['required_education'].astype('category').cat.codes

In [11]:
# Process salary range

def process_salary(salary):
    try:
        low, high = salary.split('-')
        return (int(low) + int(high)) / 2
    except:
        return None

df['salary_processed'] = df['salary_range'].apply(process_salary)

In [12]:
# Fill missing salary

df['salary_processed'] = df['salary_processed'].fillna(df['salary_processed'].median())

## 3. Feature Matrix

We select relevant features for anomaly detection.

In [13]:
features = ['salary_processed', 'employment_type', 'has_company_logo', 'required_education']

X = df[features]

X.head()

,salary_processed,employment_type,has_company_logo,required_education
0,92500.0,1,1,2
1,44000.0,1,0,9
2,44000.0,1,1,9
3,44000.0,1,1,1
4,44000.0,3,1,4


In [14]:
#Checking shape

X.shape

(8696, 4)

In [15]:
X.isnull().sum()

,0
salary_processed,0
employment_type,0
has_company_logo,0
required_education,0


## 4. Isolation Forest Model

This model detects anomalies by isolating observations.

In [16]:
from sklearn.ensemble import IsolationForest

In [17]:
#Training Model
model = IsolationForest(
    n_estimators=100,      # number of trees
    contamination=0.05,    # expected % of fraud (5%)
    random_state=42
)

model.fit(X)

IsolationForest(contamination=0.05, random_state=42)

In [18]:
#anomaly predictions

df['anomaly_score'] = model.decision_function(X)
df['anomaly'] = model.predict(X)

In [19]:
df[['salary_processed', 'anomaly_score', 'anomaly']].head()

,salary_processed,anomaly_score,anomaly
0,92500.0,0.035091,1
1,44000.0,0.174933,1
2,44000.0,0.234666,1
3,44000.0,0.236785,1
4,44000.0,0.091407,1


In [20]:
#Counting frauds

df['anomaly'].value_counts()

,count
anomaly,
1,8262
-1,434


Checking actual fraud overlap

In [21]:
df[['fraudulent', 'anomaly']].head()

,fraudulent,anomaly
0,0,1
1,0,1
2,0,1
3,0,1
4,0,1


## 5. Local Outlier Factor (LOF)

LOF identifies anomalies based on local density differences.

In [22]:
from sklearn.neighbors import LocalOutlierFactor

In [23]:
#Intializing Model

lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    novelty=True   # IMPORTANT
)

In [24]:
#Training Model

lof.fit(X)

LocalOutlierFactor(contamination=0.05, novelty=True)

In [25]:
#Getting predictions

df['lof_anomaly'] = lof.predict(X)
df['lof_score'] = lof.decision_function(X)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


In [26]:
df['lof_anomaly'].value_counts()

,count
lof_anomaly,
1,8270
-1,426


In [27]:
df[['anomaly', 'lof_anomaly']].head(10)

,anomaly,lof_anomaly
0,1,-1
1,1,1
2,1,1
3,1,1
4,1,1
5,1,1
6,1,1
7,1,1
8,1,-1
9,1,1


## 6. Model Comparison

We compare predictions from both models.

In [28]:
(df['anomaly'] == df['lof_anomaly']).sum()

np.int64(7988)

In [29]:
pd.crosstab(df['fraudulent'], df['anomaly'])
pd.crosstab(df['fraudulent'], df['lof_anomaly'])

lof_anomaly,-1,1
fraudulent,,
0,395,7879
1,31,391


## 7. Inference Functions

These functions allow scoring of new job postings.

In [30]:
def anomaly_score(row):
    import pandas as pd

    input_data = pd.DataFrame([{
        'salary_processed': row['salary_processed'],
        'employment_type': row['employment_type'],
        'has_company_logo': row['has_company_logo'],
        'required_education': row['required_education']
    }])

    score = model.decision_function(input_data)[0]

    return score

In [31]:
def anomaly_predict(row):
    import pandas as pd

    input_data = pd.DataFrame([{
        'salary_processed': row['salary_processed'],
        'employment_type': row['employment_type'],
        'has_company_logo': row['has_company_logo'],
        'required_education': row['required_education']
    }])

    return model.predict(input_data)[0]

## 8. Testing Inference

In [32]:
anomaly_score(df.iloc[0])

np.float64(0.035090606881010844)

In [33]:
anomaly_predict(df.iloc[0])

np.int64(1)

## 9. Conclusion

- Both Isolation Forest and LOF successfully identified ~5% anomalies.
- These anomalies likely represent fraudulent job postings.
- The models show consistent behavior and can be used for fraud detection.
- Future improvements include hyperparameter tuning and evaluation against labeled data.